# EDR-REDNet Training Notebook (Kaggle)
## Ablation Study: Variant B, C và D (Full EDR-REDNet)

| Variant | FixedSobelLayer | EdgeBlock | Sobel Loss | Mô tả |
|---------|----------------|-----------|------------|--------|
| **A** | ❌ | ❌ | ❌ | RED-CNN Baseline (đã có) |
| **B** | ❌ | ✅ | ❌ | Chỉ thêm EdgeBlock |
| **C** | ✅ | ✅ | ❌ | + FixedSobelLayer input |
| **D** | ✅ | ✅ | ✅ | **EDR-REDNet Full** (đã có) |

> ⚙️ **Hướng dẫn:** Chạy Cell 1-5 (Setup) → Chọn Variant cần train → Chạy Cell Training tương ứng → Chạy Cell Save Output.

In [ ]:
# ─── CELL 1: Setup môi trường ───────────────────────────────────────────────
import os

os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_START_METHOD"] = "thread"

!git clone https://github.com/minhvuongle2004/lung-diagnosis.git
%cd /kaggle/working/lung-diagnosis/ldct-benchmark
!pip install -e . -q

print("✅ Setup done!")

In [ ]:
# ─── CELL 2: Tìm đường dẫn dữ liệu ─────────────────────────────────────────
import os

print("=== /kaggle/input/ contents ===")
for item in os.listdir("/kaggle/input"):
    print(f"  {item}/")

data_path = None
print("\n=== Tìm LDCT-and-Projection-data ===")
for dataset_slug in os.listdir("/kaggle/input"):
    base = f"/kaggle/input/{dataset_slug}"
    for root, dirs, files in os.walk(base):
        if "LDCT-and-Projection-data" in dirs:
            data_path = root
            sub = os.path.join(root, "LDCT-and-Projection-data")
            patients = sorted(os.listdir(sub))
            print(f"✅ Datafolder: {data_path}")
            print(f"   Số bệnh nhân: {len(patients)}")
            print(f"   5 đầu: {patients[:5]}")
            break
    if data_path:
        break

if data_path is None:
    print("❌ Không tìm thấy LDCT-and-Projection-data!")

In [ ]:
# ─── CELL 3: Kiểm tra và lọc info.yml ───────────────────────────────────────
import os, yaml

info_path = "ldctbench/data/info.yml"
with open(info_path) as f:
    info = yaml.safe_load(f)

ldct_dir = os.path.join(data_path, "LDCT-and-Projection-data")
available = set(os.listdir(ldct_dir))

print(f"Bệnh nhân có sẵn: {len(available)}")
missing_patients = []
wrong_slices = []

for split in ["train_set", "val_set", "test_set"]:
    for entry in info.get(split, []):
        pid = entry["id"]
        if pid not in available:
            missing_patients.append((split, pid))
            continue
        input_rel = entry["input"].replace("./LDCT-and-Projection-data/", "")
        input_full = os.path.join(ldct_dir, input_rel)
        if not os.path.exists(input_full):
            wrong_slices.append((split, pid, 0, entry["n_slices"], "FOLDER MISSING"))
            continue
        actual_files = [f for f in os.listdir(input_full) if f.endswith(".dcm")]
        actual_n = len(actual_files)
        expected_n = entry["n_slices"]
        if actual_n == 0:
            wrong_slices.append((split, pid, actual_n, expected_n, "EMPTY FOLDER"))
        elif actual_n != expected_n:
            wrong_slices.append((split, pid, actual_n, expected_n, "SLICE MISMATCH"))

print(f"\n❌ Bệnh nhân thiếu: {len(missing_patients)}")
for split, pid in missing_patients[:10]:
    print(f"   [{split}] {pid}")

print(f"\n⚠️ Bệnh nhân có folder nhưng DCM thiếu: {len(wrong_slices)}")
if not missing_patients and not wrong_slices:
    print("\n✅ Tất cả dữ liệu đều OK!")

In [ ]:
# ─── CELL 4: Lọc info.yml (chỉ giữ bệnh nhân có đủ data) ────────────────────
import os, yaml, shutil

info_path = "ldctbench/data/info.yml"
with open(info_path) as f:
    info = yaml.safe_load(f)

ldct_dir = os.path.join(data_path, "LDCT-and-Projection-data")
available = set(os.listdir(ldct_dir))

new_info = {k: v for k, v in info.items() if k not in ["train_set", "val_set", "test_set"]}
for split in ["train_set", "val_set", "test_set"]:
    original = info.get(split, [])
    filtered = []
    for entry in original:
        pid = entry["id"]
        if pid not in available:
            continue
        input_rel = entry["input"].replace("./LDCT-and-Projection-data/", "")
        input_full = os.path.join(ldct_dir, input_rel)
        if not os.path.exists(input_full):
            continue
        actual_files = sorted([f for f in os.listdir(input_full) if f.endswith(".dcm")])
        if len(actual_files) == 0:
            continue
        entry = dict(entry)
        entry["n_slices"] = len(actual_files)
        filtered.append(entry)
    new_info[split] = filtered
    print(f"{split}: {len(original)} → {len(filtered)} bệnh nhân")

shutil.copy(info_path, info_path + ".bak")
with open(info_path, "w") as f:
    yaml.dump(new_info, f, default_flow_style=False, allow_unicode=True)
print(f"\n✅ Đã lưu info.yml mới")

In [ ]:
# ─── CELL 5: Patch định dạng file .dcm (nếu cần) ────────────────────────────
import os, re

ldct_dir = os.path.join(data_path, "LDCT-and-Projection-data")
sample_dcm = None

for patient in sorted(os.listdir(ldct_dir)):
    patient_path = os.path.join(ldct_dir, patient)
    for root, dirs, files in os.walk(patient_path):
        dcm_files = sorted([f for f in files if f.endswith(".dcm")])
        if dcm_files:
            sample_dcm = dcm_files[0]
            break
    if sample_dcm:
        break

print(f"Sample DCM filename: {sample_dcm}")

ldct_mayo_path = "ldctbench/data/LDCTMayo.py"
with open(ldct_mayo_path, "r") as f:
    content = f.read()

if sample_dcm and not sample_dcm.startswith("0"):
    match = re.match(r'^(.*?)(\d+)\.dcm$', sample_dcm)
    if match:
        prefix = match.group(1)
        digits = len(match.group(2))
        old_line = '        return "{}.dcm".format(str(idx).zfill(8))'
        new_line = f'        return "{prefix}{{}}.dcm".format(str(idx).zfill({digits}))'
        if old_line in content:
            content = content.replace(old_line, new_line)
            with open(ldct_mayo_path, "w") as f:
                f.write(content)
            print(f"✅ Patched LDCTMayo.py → format '{prefix}{{:0{digits}d}}.dcm'")
        else:
            print("⚠️ Format cu khong tim thay de patch.")
else:
    print(f"✅ Format da dung (00000XXX.dcm), khong can patch")

---
## TRAINING CELLS
### Chọn 1 trong 3 cell bên dưới để train Variant tương ứng

In [ ]:
# ─── CELL 6A: Train VARIANT B ────────────────────────────────────────────────
# Variant B = EdgeBlock (Dilated Residual) + KHÔNG có FixedSobelLayer + KHÔNG có Sobel Loss
# Chứng minh: Dilated residual block có giúp ích không?
import glob, os

assert data_path is not None, "❌ Chạy Cell 2 trước!"

VARIANT = "B"
seed = 1339
max_iterations = 31000  # Đủ để thấy sự hội tụ, tiết kiệm GPU hours

resume_config = ""
checkpoints = glob.glob(f"/kaggle/input/**/variant{VARIANT}_seed{seed}_best_*.pt", recursive=True)
if checkpoints:
    ckpt_path = checkpoints[0]
    resume_config = f"resume: '{ckpt_path}'"
    max_iterations = 31000
    print(f"✅ Resume từ: {ckpt_path}")
else:
    print(f"ℹ️ Train Variant B từ đầu ({max_iterations} iterations)")

config = f"""trainer: edrrednet
seed: {seed}
datafolder: {data_path}
{resume_config}
optimizer: adam
lr: 9.583e-05
adam_b1: 0.9
adam_b2: 0.999
loss_alpha: 0.0
loss_beta: 0.0
loss_gamma: 0.0
num_edge_blocks: 2
use_sobel_input: false
mbs: 16
max_iterations: {max_iterations}
data_subset: 1.0
patchsize: 128
iterations_before_val: 1000
valsamples: 8
data_norm: meanstd
num_workers: 2
cuda: true
devices: 0
"""

config_path = f"configs/ablation_variant{VARIANT}_seed{seed}.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    f.write(config)

print(f"Config saved → {config_path}")
print(f"Bat dau training Variant {VARIANT} seed {seed}...")
!python -m ldctbench.scripts.train --config {config_path}

In [ ]:
# ─── CELL 6B: Train VARIANT C ────────────────────────────────────────────────
# Variant C = EdgeBlock + FixedSobelLayer (input) + KHÔNG có Sobel Loss
# Chứng minh: Việc nạp thông tin biên vào đầu vào có giúp ích không?
import glob, os

assert data_path is not None, "❌ Chạy Cell 2 trước!"

VARIANT = "C"
seed = 1339
max_iterations = 31000

resume_config = ""
checkpoints = glob.glob(f"/kaggle/input/**/variant{VARIANT}_seed{seed}_best_*.pt", recursive=True)
if checkpoints:
    ckpt_path = checkpoints[0]
    resume_config = f"resume: '{ckpt_path}'"
    max_iterations = 31000
    print(f"✅ Resume tu: {ckpt_path}")
else:
    print(f"ℹ️ Train Variant C tu dau ({max_iterations} iterations)")

config = f"""trainer: edrrednet
seed: {seed}
datafolder: {data_path}
{resume_config}
optimizer: adam
lr: 9.583e-05
adam_b1: 0.9
adam_b2: 0.999
loss_alpha: 0.0
loss_beta: 0.0
loss_gamma: 0.0
num_edge_blocks: 2
use_sobel_input: true
mbs: 16
max_iterations: {max_iterations}
data_subset: 1.0
patchsize: 128
iterations_before_val: 1000
valsamples: 8
data_norm: meanstd
num_workers: 2
cuda: true
devices: 0
"""

config_path = f"configs/ablation_variant{VARIANT}_seed{seed}.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    f.write(config)

print(f"Config saved → {config_path}")
print(f"Bat dau training Variant {VARIANT} seed {seed}...")
!python -m ldctbench.scripts.train --config {config_path}

In [ ]:
# ─── CELL 6D: Train VARIANT D (Full EDR-REDNet) ──────────────────────────────
# Variant D = Full model: EdgeBlock + FixedSobelLayer + Sobel Loss
# Đây là mô hình chính (đã có seed 1339, chạy seed khác nếu cần)
import glob, os

assert data_path is not None, "❌ Chạy Cell 2 trước!"

VARIANT = "D"
seed = 2024  # Đổi seed nếu muốn chạy seed mới
max_iterations = 45000

resume_config = ""
checkpoints = glob.glob(f"/kaggle/input/**/seed{seed}_best_*.pt", recursive=True)
if checkpoints:
    ckpt_path = checkpoints[0]
    resume_config = f"resume: '{ckpt_path}'"
    max_iterations = 93000
    print(f"✅ Resume tu: {ckpt_path}")
    print(f"   -> Se Resume va train tiep den {max_iterations} iterations.")
else:
    print(f"ℹ️ Train Variant D tu dau ({max_iterations} iterations ~9.5h).")

config = f"""trainer: edrrednet
seed: {seed}
datafolder: {data_path}
{resume_config}
optimizer: adam
lr: 9.583e-05
adam_b1: 0.9
adam_b2: 0.999
loss_alpha: 0.1
loss_beta: 0.0
loss_gamma: 0.0
num_edge_blocks: 2
use_sobel_input: true
mbs: 16
max_iterations: {max_iterations}
data_subset: 1.0
patchsize: 128
iterations_before_val: 500
valsamples: 8
data_norm: meanstd
num_workers: 2
cuda: true
devices: 0
"""

config_path = f"configs/edrrednet_kaggle.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    f.write(config)

print(f"Config saved → {config_path}")
print(f"Bat dau training Variant {VARIANT} seed {seed}...")
!python -m ldctbench.scripts.train --config {config_path}

In [ ]:
# ─── CELL 7: Lưu Checkpoint và Log ra Output ─────────────────────────────────
# Chạy cell này SAU KHI training xong để lưu file
import glob, shutil, os

output_dir = "/kaggle/working"

# Đổi thông tin variant và seed tương ứng với cell bạn vừa chạy
VARIANT = "B"  # <── Đổi thành B, C hoặc D
seed = 1339    # <── Đổi thành seed bạn đã train

# Checkpoint
checkpoints = glob.glob("wandb/offline-run-*/files/best_*.pt")
print(f"Checkpoints found: {checkpoints}")
for ckpt in checkpoints:
    dest = os.path.join(output_dir, f"variant{VARIANT}_seed{seed}_{os.path.basename(ckpt)}")
    shutil.copy(ckpt, dest)
    print(f"✅ Saved: {dest}")

# Loss/metrics CSV
csv_files = glob.glob("wandb/offline-run-*/files/*.csv")
for csv in csv_files:
    dest = os.path.join(output_dir, f"variant{VARIANT}_seed{seed}_{os.path.basename(csv)}")
    shutil.copy(csv, dest)
    print(f"✅ Log: {dest}")

print("\n✅ Done! Click 'Save Version' de luu output.")